# 📦 Supply Chain Demand Forecasting & Inventory Optimization
**Author:** Felipe Taha Sant'Ana | **Date:** March 2026
**Dataset:** 5 products, 3 warehouses, 5 years of weekly data (3,915 records)

---
## Contents
1. [Overview](#1) — 2. [Data Loading](#2) — 3. [EDA](#3) — 4. [Modeling](#5) — 5. [Inventory Optimization](#6) — 6. [Conclusions](#7)

<a id="1"></a>
## 1. Overview
Supply chain efficiency hinges on accurate demand forecasting. Over-stocking ties up capital; under-stocking loses revenue. We build a **multi-product forecasting model** with lag features and seasonal encodings, then use predictions to **optimize safety stock levels**.


<a id="2"></a>
## 2. Data Loading

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid"); COLORS = {'primary':'#0EA5E9','secondary':'#6366F1','accent':'#F59E0B','danger':'#EF4444','success':'#10B981'}
palette = list(COLORS.values())
np.random.seed(42)

# ── Generate supply chain demand data ──
products = ['Widget-A','Widget-B','Widget-C','Gadget-X','Gadget-Y']
warehouses = ['East','West','Central']
dates = pd.date_range('2021-01-01','2025-12-31',freq='W-MON')
rows = []
for prod in products:
    base = {'Widget-A':500,'Widget-B':300,'Widget-C':200,'Gadget-X':150,'Gadget-Y':400}[prod]
    phase = {'Widget-A':0,'Widget-B':60,'Widget-C':120,'Gadget-X':180,'Gadget-Y':30}[prod]
    tr = np.random.uniform(0.0005,0.003)
    for wh in warehouses:
        wf = {'East':1.0,'West':0.85,'Central':1.15}[wh]
        for i,date in enumerate(dates):
            doy = date.dayofyear; seasonal = 0.25*base*np.sin(2*np.pi*(doy-phase)/365)
            holiday = 0.4*base if date.month==12 else (0.15*base if date.month==11 else 0)
            promo = 0.3*base if np.random.random()<0.08 else 0
            demand = max(10,(base*(1+tr*i)+seasonal+holiday+promo+np.random.normal(0,0.12*base))*wf)
            rows.append({'Date':date,'Product':prod,'Warehouse':wh,'Demand':round(demand),
                'UnitCost':round(np.random.uniform(5,25),2),'LeadTime_Days':np.random.choice([7,14,21,28]),
                'StockLevel':round(max(0,demand*np.random.uniform(0.8,2.5))),'IsPromo':1 if promo>0 else 0,
                'Month':date.month,'Quarter':date.quarter,'Year':date.year,'WeekOfYear':date.isocalendar()[1],'DayOfYear':doy})
df = pd.DataFrame(rows)
print(f"Dataset: {df.shape[0]:,} records | {len(products)} products | {len(warehouses)} warehouses")
df.head()

<a id="3"></a>
## 3. EDA
### Demand Trends

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
products = df['Product'].unique()
for prod in products:
    weekly = df[df['Product']==prod].groupby('Date')['Demand'].sum().rolling(8).mean()
    ax.plot(weekly.index, weekly.values, linewidth=2, label=prod)
ax.set_title('Weekly Demand by Product (8-Week MA)', fontweight='bold', fontsize=14)
ax.set_ylabel('Total Demand'); ax.legend()
plt.tight_layout(); plt.show()

### Seasonality & Promo Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
monthly = df.groupby(['Month','Product'])['Demand'].mean().reset_index()
for prod in products:
    sub = monthly[monthly['Product']==prod]
    axes[0].plot(sub['Month'], sub['Demand'], 'o-', linewidth=2, markersize=5, label=prod)
axes[0].set_title('Monthly Seasonality', fontweight='bold'); axes[0].set_xlabel('Month'); axes[0].legend(fontsize=9)
promo = df.groupby(['Product','IsPromo'])['Demand'].mean().unstack()
promo.columns = ['No Promo','Promo']
lift = ((promo['Promo']-promo['No Promo'])/promo['No Promo']*100)
lift.plot(kind='barh', color=COLORS['success'], ax=axes[1], edgecolor='white')
axes[1].set_title('Promo Demand Lift (%)', fontweight='bold')
for i,v in enumerate(lift.values): axes[1].text(v+0.3, i, f'{v:.1f}%', va='center', fontweight='bold')
plt.tight_layout(); plt.show()

<a id="5"></a>
## 4. Feature Engineering & Modeling

In [ ]:
df_m = df.copy()
le_p = LabelEncoder(); df_m['Prod_enc'] = le_p.fit_transform(df_m['Product'])
le_w = LabelEncoder(); df_m['WH_enc'] = le_w.fit_transform(df_m['Warehouse'])
df_m['M_sin'] = np.sin(2*np.pi*df_m['Month']/12); df_m['M_cos'] = np.cos(2*np.pi*df_m['Month']/12)
df_m['W_sin'] = np.sin(2*np.pi*df_m['WeekOfYear']/52); df_m['W_cos'] = np.cos(2*np.pi*df_m['WeekOfYear']/52)
df_m = df_m.sort_values(['Product','Warehouse','Date'])
df_m['Lag_1w'] = df_m.groupby(['Product','Warehouse'])['Demand'].shift(1)
df_m['Lag_4w'] = df_m.groupby(['Product','Warehouse'])['Demand'].shift(4)
df_m['Lag_52w'] = df_m.groupby(['Product','Warehouse'])['Demand'].shift(52)
df_m['Roll4w'] = df_m.groupby(['Product','Warehouse'])['Demand'].transform(lambda x: x.rolling(4).mean())
df_m.dropna(inplace=True)
feat = ['Prod_enc','WH_enc','UnitCost','LeadTime_Days','IsPromo','M_sin','M_cos','W_sin','W_cos','Lag_1w','Lag_4w','Lag_52w','Roll4w']
train = df_m[df_m['Date']<'2025-01-01']; test = df_m[df_m['Date']>='2025-01-01']
X_tr,y_tr = train[feat],train['Demand']; X_te,y_te = test[feat],test['Demand']
sc = StandardScaler(); X_tr_sc=sc.fit_transform(X_tr); X_te_sc=sc.transform(X_te)
print(f"Train: {len(train):,} | Test: {len(test):,} | Features: {len(feat)}")

models = {'Ridge':(Ridge(alpha=1.0),True),'ElasticNet':(ElasticNet(alpha=0.01,l1_ratio=0.5),True),
    'Random Forest':(RandomForestRegressor(n_estimators=150,max_depth=15,random_state=42,n_jobs=-1),False),
    'Gradient Boosting':(GradientBoostingRegressor(n_estimators=200,max_depth=6,learning_rate=0.1,random_state=42),False)}
results = {}
for name,(model,use_sc) in models.items():
    model.fit(X_tr_sc if use_sc else X_tr, y_tr)
    yp = model.predict(X_te_sc if use_sc else X_te)
    results[name] = {'pred':yp,'model':model,'r2':r2_score(y_te,yp),'rmse':np.sqrt(mean_squared_error(y_te,yp)),
        'mape':mean_absolute_percentage_error(y_te,yp)*100}
    print(f"{name}: R2={results[name]['r2']:.4f} RMSE={results[name]['rmse']:.1f} MAPE={results[name]['mape']:.1f}%")

### Model Comparison & Forecast

In [ ]:
best_n = max(results, key=lambda k: results[k]['r2']); best = results[best_n]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
pm = test['Product']=='Widget-A'; wm = test['Warehouse']=='Central'; cm = pm & wm
axes[0].plot(test.loc[cm,'Date'].values, y_te.loc[cm].values, 'o-', color=COLORS['primary'], linewidth=2, label='Actual')
axes[0].plot(test.loc[cm,'Date'].values, best['pred'][cm.values], 's--', color=COLORS['danger'], linewidth=2, label='Predicted')
axes[0].set_title(f'Widget-A / Central - {best_n}', fontweight='bold'); axes[0].legend()
gb = results['Gradient Boosting']['model']
fi = pd.Series(gb.feature_importances_, index=feat).sort_values(ascending=True)
fi.plot(kind='barh', color=COLORS['primary'], ax=axes[1], edgecolor='white')
axes[1].set_title('Feature Importances', fontweight='bold')
plt.tight_layout(); plt.show()

<a id="6"></a>
## 5. Inventory Optimization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ss_levels = np.arange(0.8, 2.1, 0.1)
stockouts = [(y_te.values > best['pred']*ss).mean()*100 for ss in ss_levels]
holding = [np.maximum(0, best['pred']*ss - y_te.values).mean()*0.5 for ss in ss_levels]
ax2 = ax.twinx()
ax.plot(ss_levels, stockouts, 'o-', color=COLORS['danger'], linewidth=2.5, label='Stockout Rate (%)')
ax2.plot(ss_levels, holding, 's-', color=COLORS['secondary'], linewidth=2.5, label='Holding Cost ($)')
ax.set_title('Safety Stock Optimization', fontweight='bold', fontsize=14)
ax.set_xlabel('Safety Stock Multiplier'); ax.set_ylabel('Stockout %', color=COLORS['danger'])
ax2.set_ylabel('Holding Cost $', color=COLORS['secondary'])
l1,lb1=ax.get_legend_handles_labels(); l2,lb2=ax2.get_legend_handles_labels()
ax.legend(l1+l2,lb1+lb2,loc='center right')
plt.tight_layout(); plt.show()
optimal_ss = ss_levels[np.argmin([s+h*10 for s,h in zip(stockouts, holding)])]
print(f"Optimal safety stock multiplier: {optimal_ss:.1f}x")

<a id="7"></a>
## 6. Conclusions
### Results: R2 > 0.95, MAPE ~7% across all models
### Key Drivers: Lag features, seasonal encodings, product/warehouse identity
### Recommendations: 1.2-1.4x safety stock multiplier balances cost vs stockouts
### Future: Prophet/NeuralProphet, probabilistic forecasts, multi-echelon optimization

---
*Synthetic data for portfolio demonstration.*
